### This notebook shall be used to train base forecasting models across the hierarchy and test them on a holdout. I'll be using Naive, SeasonalNaive, AutoARIMA and ETS models to train and test the models. The models will be evaluated for MASE and RMSE metrics

In [ ]:
# Importing the libraries
import pandas as pd
import numpy as np

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoARIMA, AutoETS

In [ ]:
pd.set_option('display.max_columns',None)
pd.set_option('display.width',200)

In [ ]:
# Loading the processed dataset
df = pd.read_csv('../data/processed/hierarchy_ready.csv')

In [ ]:
# Ensuring the 'ds' column is in the right format
df['ds'] = pd.to_datetime(df['ds'])

In [ ]:
print("Shape: ",df.shape)

In [ ]:
print("Unique Series: ",df['unique_id'].nunique())

In [ ]:
print("Date Range: ",df['ds'].min(), " to ",df['ds'].max())

#### For the train and test split, the last 12 months of data shall be used as holdout for testing the models. Since the data is available only till Feb 2026, the 12 month data, from March 2025 to Feb 2026 shall be used as the test data 

In [ ]:
split_date = pd.Timestamp('2025-03-01')

train = df[df['ds'] < split_date].copy()
test = df[df['ds']>=split_date].copy()

In [ ]:
print("Train range: ",train['ds'].min(), " to ", train['ds'].max())
print("Train Rows: ",len(train))

In [ ]:
print("Test range: ",test['ds'].min(), " to ", test['ds'].max())
print("Test Rows: ",len(test))

In [ ]:
print("Test months per series: ",test.groupby('unique_id').size().describe()[['min','max','mean']].to_dict())

#### There are 232 series, so the test series should have 232 x 12 = 2784 rows. Here there are only 2736 rows. In other words, there is a shortage of 48 rows or 4 series

In [ ]:
test_counts = test.groupby('unique_id').size()
print("Series in test: ",len(test_counts))

print("Min: ",test_counts.min(), "Max: ",test_counts.max())

print("Series < 12: ",(test_counts < 12).sum())

print("Total series in full dataframe: ",df['unique_id'].nunique())


#### The above output confirms the fact that 4 series are completly missing from the test data. Routes appear in the training data, bu are missing in test data. The coherence of the hierarchy is compromised here as there is no sufficient test data to evaluate. Mintrace reconciliation as requires a complete, coherent hierarchy.

In [ ]:
# Find the missing 4 series from the test data
all_series = set(df['unique_id'].unique())

test_series = set(test['unique_id'].unique())

missing_series = all_series - test_series

print("Series missing from the test data set: ")
for s in missing_series:
    last_date = df[df['unique_id'] == s]['ds'].max()
    level = s.count('/') # 0 - Total, 1 - Airport, 2 - Route
    print(f"{s} | last data: {last_date.date()} | level: {level}")

In [ ]:
# handling missing values by dropping them
to_drop = {
    "TOTAL/MUMBAI/MUMBAI-RAJKOT",
    "TOTAL/DELHI/DELHI-RAJKOT",
    "TOTAL/DEHRA DUN/DEHRA DUN-DELHI",
    "TOTAL/DEHRA DUN"
}

# We need to check if any of the airports lose all its routes after the above routes are dropped
remaining = df[~df['unique_id'].isin(to_drop)].copy()

remaining_routes = remaining[remaining['unique_id'].str.count('/') == 2].copy()

remaining_routes['origin'] = remaining_routes['unique_id'].str.split('/').str[1]

routes_per_airport = remaining_routes['origin'].nunique()

print("Unique origin airports still having routes: ",routes_per_airport)
print("Airport-level series remaining: ",remaining[remaining['unique_id'].str.count('/')==1]['unique_id'].nunique())

In [ ]:
# Applying the drop to the original dataset and repeating the train/test split

df = df[~df['unique_id'].isin(to_drop)].copy()

split_date = pd.Timestamp('2025-03-01')

train = df[df['ds']<split_date].copy()
test = df[df['ds']>=split_date].copy()


# Crosschecking the numbers for confirmation

print("Total series after the drop: ", df['unique_id'].nunique())
print("Series in test dataset: ", test['unique_id'].nunique())

print("Train dataset range: ",train['ds'].min().date()," to ",train['ds'].max().date())
print("Test dataset range: ",test['ds'].min().date(), " to ",test['ds'].max().date())

In [ ]:
# Check for hierarchy breakdown

print(df['unique_id'].str.count('/').value_counts().to_dict())

levels = df.drop_duplicates('unique_id')['unique_id'].str.count('/').value_counts().to_dict()
print("Levels (0 - Total, 1 - Airport, 2 - Route): ",levels)


In [ ]:
# Initiate the time series models

models = [
    Naive(),
    SeasonalNaive(season_length = 12),
    AutoARIMA(season_length = 12),
    AutoETS(season_length = 12)
]

# Forecasting Engine with parameters
sf = StatsForecast(
    models = models,
    freq = 'MS', # Monthly Start Frequency
    n_jobs = -1, # Use all CPU cores in parallel
    verbose = True
)

# Fit on training data
sf.fit (df = train)

print("All models fitted across ",train['unique_id'].nunique(), "series")

In [ ]:
# Generate 12-month forecasts for the test data
forecasts = sf.predict(h = 12)

print("Forecast shape: ",forecasts.shape)
print(forecasts.head())
print("Columns: ",forecasts.columns.tolist())

In [ ]:
# Merge forecasts on to the test set (actual numbers)
eval_df = test.merge(forecasts, on = ['unique_id','ds'], how = 'left')

# Sanity Checks
print("Shape: ",eval_df.shape)
print("Missing forecasts: ",eval_df[['Naive','SeasonalNaive','AutoARIMA','AutoETS']].isna().sum().sum())
print("Sample rows: ")
print(eval_df.head(5))

In [ ]:
# Define Root Mean Squared Error (RMSE) function

def rmse (y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

# Define Mean Absolute Scaled Error (MASE) function

def mase (y_true, y_pred, y_train, season_length = 12):
    naive_errors = np.abs(y_train[season_length:] - y_train[:-season_length])
    scale = naive_errors.mean()

    if scale == 0:
        return np.nan
    return np.mean(np.abs(y_true - y_pred) / scale)


#### RMSE is easy to interpret on a single series, but is meaningless when used to compare across hierarchies. MASE is comparable across hierarchies. It is the right primary metric when we are aggregating across a hierarchy

In [ ]:
model_cols = ['Naive','SeasonalNaive','AutoARIMA','AutoETS']

# Training history for MASE to scale
train_history = train.groupby('unique_id')['y'].apply(lambda s: s.values).to_dict()

records = []
for uid, g in eval_df.groupby('unique_id'):
    y_true = g['y'].values
    y_train = train_history[uid]
    for m in model_cols:
        y_pred = g[m].values
        records.append({
            'unique_id':uid,
            'model':m,
            'rmse':rmse(y_true,y_pred),
            'mase':mase(y_true, y_pred, y_train, season_length = 12)
        })

metrics_df = pd.DataFrame(records)

print("Evaluation Metrics computed for: ", len(metrics_df), " rows")
print("Sample: ")
print(metrics_df.head())

In [ ]:
# Aggregate metrics by hierrachy level

metrics_df['level'] = metrics_df['unique_id'].apply(
    lambda x: 'TOTAL' if '/' not in x
    else ('Airport' if x.count('/') == 1 else 'Route') 
)

# Aggregate average MASE and RMSE per hierarchy level

summary = metrics_df.groupby(['level','model'])[['mase','rmse']].mean().round(3).reset_index()

# Pivotting for readability

mase_table = summary.pivot(index = 'model', columns = 'level', values = 'mase')[['TOTAL','Airport','Route']]
rmse_table = summary.pivot(index = 'model', columns = 'level', values = 'rmse')[['TOTAL','Airport','Route']]

print("Average MASE by hierarchy level: ")
print(mase_table)

print("Average RMSE by hierarchy level: ")
print(rmse_table)


#### The evaluation metrics aggregated at hierarchy levels painted a different story than expected. There was no single model which topped at all levels. While SeasonalNaive led the Total and Airport levels, AutoARIMA took the lead at Route level. AutoETS finished a very close second at this level. 

#### The relatively better performance of SeasonalNaive model can be attributed to the strong and consistent seasonality present in the data even with the pandemic being considered.

#### At the individual Route level (without any aggregation), the performance of AutoARIMA model reveals the autoregressive nature of the series dominating seasonality at granular level. 

#### Since there is no single model consistent across all three levels, the base forecasts won't aggregate coherently. For the next step, MinTrace reconciliation will combine the forecasts across levels and solve the problem arising from using different forecasting models at different hierarchy levels.

#### The AutoETS forecast results underperformed at the Airport level and were near-flat at the TOTAL level. This is most likely a  consequence of the automatic ETS selector choosing degenerate configurations due to pandemic effect, but needs to be ascertained with further analysis before reaching a conclusion.

In [ ]:
# Save the base forecasts
forecasts.to_csv('../outputs/base_forecasts.csv', index = False)

# Save the evaluation metrics per series
metrics_df.to_csv('../outputs/base_metrics.csv', index = False)

# Save the summary table
summary.to_csv('../outputs/base_metrics_summary.csv', index = False)

print("Output files saved: ")
print("base_forecasts.csv - ",forecasts.shape)
print("base_metrics.csv - ",metrics_df.shape)
print("base_metrics_summary.csv - ",summary.shape)